In [0]:
BRONZE_TABLE = "retail_de_27.bronze.orders"

bronze_df = spark.table(BRONZE_TABLE)


mandatory_columns = [
    "order_id",
    "customer_id",
    "product_id",
    "quantity",
    "price",
    "order_date"
]

silver_df = bronze_df.dropna(
    subset=mandatory_columns
)

print("Bronze count:", bronze_df.count())
print("After null handling:", silver_df.count())

display(silver_df)


In [0]:

from pyspark.sql.window import Window
from pyspark.sql import functions as F

window_spec = (
    Window
    .partitionBy("order_id")
    .orderBy(
        F.col("ingestion_time").desc()
    )
)

silver_df = (
    silver_df
    .withColumn(
        "row_num",
        F.row_number().over(window_spec)
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

print("After deduplication:", silver_df.count())

display(silver_df)


In [0]:

# Create Silver schema
CATALOG = "retail_de_27"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver"
)

# Write cleaned data to Silver Delta table
SILVER_TABLE = f"{CATALOG}.silver.orders"

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)

print("Silver table created successfully")


In [0]:
saved_df = spark.table(SILVER_TABLE)

display(saved_df)

print("Total records:", saved_df.count())